# 02. ProGAN 가짜 얼굴 40,000장 생성 · FFHQ 실제 얼굴 40,000장 수집

학습 데이터셋 8만 장을 직접 만든 노트북.

**왜 ProGAN인가** — 초기에는 StyleGAN2/3 생성 이미지로 학습했지만 Mean Acc 51.7%에 그쳤다.
StyleGAN의 Style Mixing과 AdaIN이 아티팩트를 해당 모델 고유 패턴으로 만들기 때문이다.
반면 ProGAN의 Progressive Growing은 해상도를 단계적으로 키우면서 업샘플링 아티팩트를
**규칙적으로** 남기기 때문에, 다른 생성 모델에도 공통으로 나타나는 패턴을 학습할 수 있다.

**막힌 지점과 우회** — NVlabs 공식 ProGAN 저장소와 CDN이 모두 접근 거부(403)를 반환해,
Meta의 `pytorch_GAN_zoo`에 공개된 동일 계열 사전학습 모델(celebAHQ-512)로 대체했다.
PyTorch 버전 차이로 `Adam` 옵티마이저가 `betas` 타입 검사에서 실패해, 해당 클래스를
임시로 감싸 float으로 강제 변환한 뒤 로드하고 바로 원복했다.

In [ ]:
# 1. NVlabs 공식 저장소 시도 → 접근 거부로 실패 (기록용)
!git clone https://github.com/NVlabs/progressive_growing_of_gans.git
%cd progressive_growing_of_gans
!wget https://nvlabs-fi-cdn.nvidia.com/progressive_growing_of_gans/karras2018iclr-celebahq-1024x1024.pkl

from google.colab import drive
drive.mount('/content/drive')

Cloning into 'progressive_growing_of_gans'...
fatal: could not read Username for 'https://github.com': No such device or address
HTTP request sent, awaiting response... 403 Forbidden
2026-05-02 00:38:36 ERROR 403: Forbidden.
Mounted at /content/drive

In [ ]:
import torch
import torchvision
import os
from tqdm.auto import tqdm

# --- PyTorch 버전 차이로 betas 타입 검사에 걸려서, 로드 동안만 Adam을 감싼다 ---
original_adam = torch.optim.Adam

class PatchedAdam(original_adam):
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8, *args, **kwargs):
        betas = (float(betas[0]), float(betas[1]))
        super().__init__(params, lr=lr, betas=betas, eps=eps, *args, **kwargs)

torch.optim.Adam = PatchedAdam
# -----------------------------------------------------------------------

# 압축 해제 시 바로 학습 구조가 되도록 로컬 디스크에 뼈대를 만든다
save_dir = '/content/train/1_fake/ProGAN'
os.makedirs(save_dir, exist_ok=True)

use_gpu = torch.cuda.is_available()
model = torch.hub.load('facebookresearch/pytorch_GAN_zoo:hub',
                       'PGAN', model_name='celebAHQ-512',
                       pretrained=True, useGPU=use_gpu)

torch.optim.Adam = original_adam  # 원복

num_images = 40000
batch_size = 16
num_batches = num_images // batch_size

img_count = 0
for i in tqdm(range(num_batches), desc="이미지 생성 진행률"):
    noise, _ = model.buildNoiseData(batch_size)
    with torch.no_grad():
        generated_images = model.test(noise)
    for j in range(batch_size):
        file_path = os.path.join(save_dir, f"progan_{img_count:05d}.png")
        torchvision.utils.save_image(generated_images[j], file_path,
                                     normalize=True, value_range=(-1, 1))
        img_count += 1

# Drive 업로드는 파일 단위로 하면 매우 느려서, 한 덩어리로 묶어서 올린다
!tar -czf /content/drive/MyDrive/WayV_Detection_Train/progan_40k.tar.gz -C /content/ train

Using cache found in /root/.cache/torch/hub/facebookresearch_pytorch_GAN_zoo_hub
Downloading: "celebaHQ16_december_s7_i96000-9c72988c.pth" (264M)
Average network found !
로컬 디스크 [/content/train/1_fake/ProGAN]에 40000장 생성 시작
이미지 생성 진행률: 100% 2500/2500 [1:24:26<00:00, 1.97s/it]
4만 장 생성 완료. 구글 드라이브로 압축 전송 시작.

In [ ]:
import os

file_path = '/content/drive/MyDrive/WayV_Detection_Train/progan_40k.tar.gz'

if os.path.exists(file_path):
    size_gb = os.path.getsize(file_path) / (1024 ** 3)
    print(f"파일 생성 확인 (용량: {size_gb:.2f} GB)")
    from google.colab import drive
    drive.mount('/content/drive')
    drive.flush_and_unmount()   # Drive 동기화 강제
else:
    print("경로에 파일이 없습니다.")

파일 생성 확인 (용량: 12.57 GB)
구글 드라이브 강제 동기화 완료.

In [ ]:
!pip install datasets -q

import os
from datasets import load_dataset
from tqdm.auto import tqdm

save_dir = '/content/train/0_real'
os.makedirs(save_dir, exist_ok=True)

# NVIDIA 공식 FFHQ 배포가 막혀 있어 HuggingFace 미러를 사용
dataset = load_dataset("marcosv/ffhq-dataset", split="train[:40000]")

for i, item in enumerate(tqdm(dataset, desc="이미지 로컬 저장 진행률")):
    img = item['image']
    if img.mode != 'RGB':
        img = img.convert('RGB')
    img.save(os.path.join(save_dir, f"real_{i:05d}.png"))

# Fake와 섞이지 않도록 0_real 폴더만 단독으로 압축
!tar -czf /content/drive/MyDrive/WayV_Detection_Train/ffhq_40k_real.tar.gz -C /content/ train/0_real

HuggingFace에서 FFHQ 실제(Real) 데이터 수집 중...
로컬 디스크에 40000장 저장
ProGAN과 분리된 Real 4만 장 압축 저장 완료.